# Functions for Each Step

In [ ]:
import numpy as np
import pandas as pd 
import time
import os
import io
from scipy.io import arff

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from llmfilter_dh import describe_data, run_filter_chunked
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)   # 5-fold-cross validation

import re
############################ Data feature names cleaning ############################
# This function cleans 'thoughts ?' -> 'thoughts?' and handles extra spaces
def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # 1. Remove leading/trailing whitespace
        c = col.strip()
        # 2. Replace multiple spaces with a single space
        c = re.sub(r'\s+', ' ', c)
        # 3. Remove space before punctuation (e.g., " ?" -> "?")
        c = re.sub(r'\s+([?.!,])', r'\1', c)
        new_columns.append(c)
    
    df.columns = new_columns
    return df

############################ Step 1: Identify Feature Types and Numericalize ############################
def identify_feature_types(df, t=15):
    metadata_list = []
    
    for col in df.columns:
        unique_vals = np.sort(df[col].unique())
        n_unique = len(unique_vals)
        dtype = df[col].dtype
        
        # --- Logic Gates for Categorization ---
        if n_unique <= t:
            # All low-cardinality features are treated as Categorical (Snapping required)
            category = 'Categ'
        
        else:
            if np.issubdtype(dtype, np.integer):
                # High-cardinality integers (Age, Study Hours)
                category = 'Num_disc'
            elif dtype == 'object':
                # High-cardinality strings (Names, IDs, Addresses)
                category = 'REMOVE'
            else:
                category = 'Num_cont' # Fallback for unknown numeric types
        
        metadata_list.append({
            'Feature': col,
            'Num Unique': n_unique,
            'Data Type': dtype,
            'Valid Values': unique_vals.tolist(),
            'Category': category
        })
    metadata = pd.DataFrame(metadata_list)
    Categ = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Categ']
    Num_disc = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_disc']
    Num_cont = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_cont']
    REMOVE = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'REMOVE']
        
    return metadata, Categ, Num_disc, Num_cont, REMOVE

def encode_to_numerical(df_org, REMOVE, Categ):
    # 1. Remove features in the REMOVE list
    df_cleaned = df_org.drop(REMOVE, axis=1)

    # 2. Initialize the translation dictionary (The "Rosetta Stone")
    # We will need this for Step 5 and Step 7
    mapping_dict = {}

    df_num = df_cleaned.copy()

    for col in Categ:
        # Get the valid values we stored in metadata
        # We sort them to ensure the mapping is consistent (e.g., 0 is always the same thing)
        unique_labels = sorted(df_num[col].unique())
        if col == Categ[-1]:
            # label = last column (always, Major = 0, Minor = 1)
            unique_labels = [df_num.iloc[:,-1].value_counts().index[0], df_num.iloc[:,-1].value_counts().index[1]]

        # Create the Forward (Text -> Num) and Reverse (Num -> Text) maps
        forward_map = {label: i for i, label in enumerate(unique_labels)}
        reverse_map = {i: label for i, label in enumerate(unique_labels)}

        # Save to our dictionary
        mapping_dict[col] = {
            'forward': forward_map,
            'reverse': reverse_map
        }

        # Apply the transformation
        df_num[col] = df_num[col].map(forward_map)

    # 3. Ensure all columns are numeric types (float or int) for SMOTE
    # This converts any remaining "object" types that might have slipped through
    df_num = df_num.apply(pd.to_numeric)
    
    return mapping_dict, df_num

############################ Step 2: Extra SMOTE generates new data points with Extrapolation ############################
def smote_generate(data, n_samples, delta_range=(0.0, 1.0), seed=None):
    # Set the seed for reproducibility if a number is provided
    # If seed is None, it uses the system clock (different every time)
    smote_k = 5                                                      # for knn in smote or extra smote
    rng = np.random.default_rng(seed)
    knn = NearestNeighbors(n_neighbors=smote_k).fit(data)            # Find k nearest neighbors for each minority point
    synthetic = []
    for _ in range(n_samples):
        idx = rng.integers(0, len(data))                             # Pick a random index from minority points
        origin = data[idx]                                           # the chosen minority point
        neighbor_distances, neighbor_indices = knn.kneighbors([origin])
        neighbor = data[rng.choice(neighbor_indices[0][1:])]         # Pick a random neighbor (excluding itself)  
        delta = rng.uniform(delta_range[0], delta_range[1])          # The random factor
        synthetic.append(origin + delta * (neighbor - origin))       # Formula: x_new = x_i + delta * (x_neighbor - x_i)
    return pd.DataFrame(synthetic, columns = df_num.columns[:-1])

############################ Step 3: Snap & Round (correction for original-like data) ############################
def process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, meta_df):
    """
    Snaps and rounds raw synthetic data based on feature classifications.
    Input df_raw_extra is a DataFrame.
    """
    df_extra = df_raw_extra.copy()
    
    # FIX: Ensure 'Feature' is the index so we can look up by name
    # We check if 'Feature' is a column; if so, we set it as index.
    if 'Feature' in meta_df.columns:
        meta_df = meta_df.set_index('Feature')
    
    for col in df_extra.columns:
        if col in Num_cont:
            continue
            
        elif col in Num_disc:
            df_extra[col] = df_extra[col].round().astype(int)
            
        elif col in Categ:
            # Look up the number of categories
            # Note: Ensure this matches the column name in your metadata ('Num Unique')
            num_categories = meta_df.loc[col, 'Num Unique']
            
            # Valid options are the integer indices 0, 1, ..., n-1
            valid_indices = np.arange(num_categories)
            
            def snap_to_closest(val):
                # Using absolute difference to find the nearest integer label
                idx = (np.abs(valid_indices - val)).argmin()
                return valid_indices[idx]
            
            df_extra[col] = df_extra[col].apply(snap_to_closest).astype(int)

    return df_extra

############################ Step 4: ENN filtering ############################
def enn_filter(synthetic_points, orig_min, orig_maj):
    enn_k = 3                                                 # for knn in enn
    X_ref = np.vstack([orig_min, orig_maj])                   # Original whole data
    y_ref = np.array([1]*len(orig_min) + [0]*len(orig_maj))   # Original data's label
    knn = NearestNeighbors(n_neighbors=enn_k).fit(X_ref)     
    
    kept, removed = [], []
    for p in synthetic_points:                                # For each extra-generated point
        distances, indices = knn.kneighbors(p.reshape(1, -1)) # Find its neighbors' indices
        # If majority of neighbors in original data are minority class, KEEP.
        if np.sum(y_ref[indices[0]]) > (enn_k / 2):
            kept.append(p)
        else:
            removed.append(p)
    return pd.DataFrame(kept, columns = df_num.columns[:-1]), pd.DataFrame(removed, columns = df_num.columns[:-1])

############################ Step 5: Decoding to the original data form with 'mapping_dict' ############################
def decode_to_original(df_after_enn, Categ, Num_disc, mapping_dict):
    """
    Translates numericalized categorical data back to original strings
    and ensures discrete numbers are properly formatted.
    """
    # Create a copy to preserve the numerical version for later
    df_after_enn_org = df_after_enn.copy()
    
    # 1. Decode Categorical Features using the "Rosetta Stone" (mapping_dict)
    for col in Categ:
        if col in mapping_dict:
            # Use the 'reverse' map: {0: 'Male', 1: 'Female'}
            reverse_map = mapping_dict[col]['reverse']
            df_after_enn_org[col] = df_after_enn_org[col].map(reverse_map)
            
    # 2. Ensure Discrete Numbers (Age, Hours) are Integers for readability
    # (Sometimes floating point noise makes them 24.0 instead of 24)
    for col in Num_disc:
        df_after_enn_org[col] = df_after_enn_org[col].astype(int)
        
    return df_after_enn_org

############################ Step 6: LLM filtering ############################
from llmfilter_dh import describe_data, run_filter_chunked

############################ Step 7: Re-Numericalization (for further experiments) ############################
def renumericalize_final_data(df_final_org, Categ, Num_disc, Num_cont, mapping_dict):
    """
    Converts LLM-approved natural language data back into numerical format.
    """
    df_final_num = df_final_org.copy()
    
    # 1. Re-encode Categorical Features using the 'forward' map
    for col in Categ:
        if col in mapping_dict:
            # Use the 'forward' map: {'Male': 0, 'Female': 1}
            forward_map = mapping_dict[col]['forward']
            df_final_num[col] = df_final_num[col].map(forward_map)
            
    # 2. Ensure Discrete columns are integers
    for col in Num_disc:
        df_final_num[col] = pd.to_numeric(df_final_num[col]).astype(int)
        
    # 3. Ensure Continuous columns are floats
    for col in Num_cont:
        df_final_num[col] = pd.to_numeric(df_final_num[col]).astype(float)
        
    return df_final_num

# Dataset Preparation

In [ ]:
final_list = ['Wine', 'Depression', 'dmft-all', 'tourism-23457vs01', 
              'diu-ro10-cat', 'newthyroid1', 'page_blocks_1_3_vs_4']
path = './Dataset/'
d_name = 'Wine'           # Choose one dataset and change for others

In [ ]:
if d_name == 'Wine' or d_name == 'Depression':
    df_org = pd.read_csv(path + str(d_name) +'.csv')                                   # csv files
else:
    with open(os.path.join(path+'Small/', str(d_name)+'.arff'), 'r') as f:         # arff files
        content = f.read().replace(', ', ',').replace(' ,', ',')
    rawdata, meta = arff.loadarff(io.StringIO(content))
    df_org = pd.DataFrame(rawdata)
    df_org = df_org.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
df_org = clean_column_names(df_org)                                                         # Clean Feature Names
df_org

In [ ]:
df_org.iloc[:,-1].value_counts()

In [ ]:
metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df_org, t=15)
print(f"Categ = {Categ}\nNum_disc = {Num_disc}\nNum_cont = {Num_cont}\nREMOVE = {REMOVE}")
metadata

In [ ]:
# # Manually Modifying (page_blocks_1_3_vs_4)
# Categ = ['Class']
# Num_disc = ['Height', 'Lenght', 'Area', 'Blackpix', 'Blackand', 'Wb_trans']
# Num_cont = ['Eccen', 'P_black', 'P_and', 'Mean_tr']

In [ ]:
# # Manually Modifying (newthyroid1)
# Categ = ['Class']
# Num_disc = ['T3resin']
# Num_cont = ['Thyroxin', 'Triiodothyronine', 'Thyroidstimulating', 'TSH_value']

In [ ]:
mapping_dict, df_num = encode_to_numerical(df_org, REMOVE, Categ)
print("--- Step 1 Complete --- \ndf_num (numericalized version) / 3 lists [Num_cont, Num_disc, Categ] / mapping_dict ")
df_num

In [ ]:
df_num.iloc[:,-1].value_counts()

# 1-1) SMO/BSM/ADA - 5CV (cross-validation train datasets) - 56% * 5

In [ ]:
# SMOTE

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "SMOTE_{}".format(Strategy[j]), "==========")
    over = SMOTE(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            X_train, y_train = over.fit_resample(X_train, y_train) 
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_cv.append(elapsed)
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/SMO/{}_SMO_{}_{}th.csv".format(d_name,d_name,Strategy[j],n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/SMO/{}_SMO_time.csv".format(d_name,d_name), index=False)

In [ ]:
# Borderline SMOTE

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "B-SMOTE_{}".format(Strategy[j]), "==========")
    over = BorderlineSMOTE(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            X_train, y_train = over.fit_resample(X_train, y_train) 
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_cv.append(elapsed)
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/BSM/{}_BSM_{}_{}th.csv".format(d_name,d_name,Strategy[j], n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/BSM/{}_BSM_time.csv".format(d_name,d_name), index=False)

In [ ]:
# ADASYN

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "ADASYN_{}".format(Strategy[j]), "==========")
    over = ADASYN(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            try:
                X_train, y_train = over.fit_resample(X_train, y_train)
                elapsed = time.time() - start
                print(f"Run took {elapsed:.2f}s")
                time_cv.append(elapsed) # Successful run 
            except Exception as e:
                print(f"Error in fold {n_iter}: {e}")
                time_cv.append(0) # <--- ADD THIS: Keep the list length consistent
                continue
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/ADA/{}_ADA_{}_{}th.csv".format(d_name,d_name,Strategy[j], n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/ADA/{}_ADA_time.csv".format(d_name,d_name), index=False)

# 1-2) SMO/BSM/ADA - Train dataset 70%

In [ ]:
# SMOTE

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "SMOTE_{}".format(Strategy[j]), "==========")
    over = SMOTE(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append(0)
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        X_over, y_over = over.fit_resample(X, y)  
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")
        time_str.append(elapsed)
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/SMO/{}_SMO_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)

In [ ]:
# Borderline SMOTE

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "B-SMOTE_{}".format(Strategy[j]), "==========")
    over = BorderlineSMOTE(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append(0)
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        X_over, y_over = over.fit_resample(X, y)  
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")
        time_str.append(elapsed)
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/BSM/{}_BSM_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)

In [ ]:
# ADASYN

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "ADASYN_{}".format(Strategy[j]), "==========")
    over = ADASYN(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append(0)
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        try:
            X_over, y_over = over.fit_resample(X, y)
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_str.append(elapsed)
        except Exception as e:
            print(f"Error: {e}")
            time_str.append(0) # <--- ADD THIS: Keep the list length consistent
            continue
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/ADA/{}_ADA_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)

# 2. XEL - 5CV (cross-validation train datasets) - 56% * 5

In [ ]:
# Need_Num factor can be different for each dataset, but for one dataset, it should be consistent
needed_num_factor = 3
# For LLM, original_sample_size & filtering_chunk_size
size = 10

In [ ]:
# chage delta_values in [1.5, 2.0, 2.5, 3.0]
delta_value = 1.5      # for extra smote, delta range = (1, delta_value)

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
yield_str = []
kept_str = []
for j in range(len(Strategy)):
    print("==========", "XEL_{}".format(Strategy[j]), "==========")
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        yield_str.append([0,0,0,0,0])
        kept_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        yield_cv = []
        kept_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)

            # Minority Generation
            orig_data = df_org                                      # Reference for LLM
            majority_class = np.array(majority_data.iloc[:,:-1])    # majority data - features 
            minority_class = np.array(minority_data.iloc[:,:-1])    # minority data - features
            df_after_llm_org = pd.DataFrame()                       # LLM_kept_DataFrame  
            current_loop_seed = 0                                   # Start our seed at 0

            time.sleep(60)
            start = time.time()
            data_desc, column_desc = describe_data(orig_data, 100)  # Get data description once to save API calls
            while len(df_after_llm_org) < needed_num:
                n_to_gen = needed_num * needed_num_factor    # target generation (3~10 times of needed_num)

                # step2. Generate with Extrapolation
                df_raw_extra = smote_generate(minority_class, n_samples=n_to_gen, delta_range=(1, delta_value), 
                                           seed=current_loop_seed)   # Every repeat, it has different random seed
                # step3. snap & round
                df_extra = process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, metadata) 
                if current_loop_seed == 0:
                    print("Extrapolation-Generated:", len(df_extra))

                # step4. Remove with ENN
                enn_kept, enn_removed = enn_filter(np.array(df_extra), minority_class, majority_class)
                if len(enn_kept) > 0:
                    df_after_enn = enn_kept.copy()
                    df_after_enn[df_num.columns[-1]] = 1
                    if current_loop_seed == 0:
                        print("Kept after ENN:", len(df_after_enn))

                    # step5. Decoding to the original form
                    df_after_enn_org = decode_to_original(df_after_enn, Categ, Num_disc, mapping_dict)

                    # step6. Remove with LLM ("arcee-ai/trinity-large-preview:free")
                    kept_index = run_filter_chunked(
                        df_after_enn_org,
                        orig_data,
                        data_desc,
                        column_desc,
                        original_sample_size=size,
                        filtering_chunk_size=size)
                    if kept_index is not None and len(kept_index) > 0:
                        sub_gen_data = df_after_enn_org.loc[kept_index]
                        df_after_llm_org = pd.concat([df_after_llm_org, sub_gen_data], axis=0)

                        # step7. Re-Numericalization
                        df_after_llm = renumericalize_final_data(df_after_llm_org, Categ, Num_disc, Num_cont, mapping_dict)
                    else:
                        print(f"Warning: Seed {current_loop_seed} produced no LLM-approved points.")
   
                print(f"Random Seed {current_loop_seed} complete. Total Kept: {len(df_after_llm_org)}/{needed_num}")    
                current_loop_seed += 1
        
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")

            # Final Result
            new_data = df_after_llm.iloc[:needed_num].reset_index(drop=True)
            time_cv.append(elapsed) 
            yield_cv.append(len(df_after_llm)/(len(df_extra)*current_loop_seed))
            kept_cv.append(len(df_after_llm))
            over_df = pd.concat([df_train, new_data], axis=0)
            over_df.to_csv("Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_{}_{}th.csv".format(d_name,delta_value,d_name, Strategy[j], n_iter), index=False)
        time_str.append(time_cv)
        yield_str.append(yield_cv)
        kept_str.append(kept_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_yield = pd.DataFrame(np.array(yield_str).T)
df_kept = pd.DataFrame(np.array(kept_str).T)
df_time.to_csv("Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_time.csv".format(d_name,delta_value,d_name), index=False)
df_yield.to_csv("Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_yield.csv".format(d_name,delta_value,d_name), index=False)
df_kept.to_csv("Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_kept.csv".format(d_name,delta_value,d_name), index=False)

# XEL - Train dataset 70%

In [ ]:
# chage delta_values in [1.5, 2.0, 2.5, 3.0]
delta_value = 1.5      # for extra smote, delta range = (1, delta_value)

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
print('<Train-Data Class Dist>\n', df_val.iloc[:,-1].value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(df_val.iloc[:,-1].value_counts()[1]/df_val.iloc[:,-1].value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((df_val.iloc[:,-1].value_counts()[1]/df_val.iloc[:,-1].value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
yield_str = []
kept_str = []
for j in range(len(Strategy)):
    print("==========", "XEL_{}".format(Strategy[j]), "==========")
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        yield_str.append([0,0,0,0,0])
        kept_str.append([0,0,0,0,0])
        continue         
    else:     
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        orig_data = df_org                                      # Reference for LLM
        majority_class = np.array(majority_data.iloc[:,:-1])    # majority data - features 
        minority_class = np.array(minority_data.iloc[:,:-1])    # minority data - features
        df_after_llm_org = pd.DataFrame()                       # LLM_kept_DataFrame  
        current_loop_seed = 0                                   # Start our seed at 0

        start = time.time()
        data_desc, column_desc = describe_data(orig_data, 100)  # Get data description once to save API calls
        while len(df_after_llm_org) < needed_num:
            n_to_gen = needed_num * needed_num_factor    # target generation (5~10 times of needed_num)

            # step2. Generate with Extrapolation
            df_raw_extra = smote_generate(minority_class, n_samples=n_to_gen, delta_range=(1, delta_value), 
                                       seed=current_loop_seed)   # Every repeat, it has different random seed
            # step3. snap & round
            df_extra = process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, metadata) 
            if current_loop_seed == 0:
                print("Extrapolation-Generated:", len(df_extra))

            # step4. Remove with ENN
            enn_kept, enn_removed = enn_filter(np.array(df_extra), minority_class, majority_class)
            if len(enn_kept) > 0:
                df_after_enn = enn_kept.copy()
                df_after_enn[df_num.columns[-1]] = 1
                if current_loop_seed == 0:
                    print("Kept after ENN:", len(df_after_enn))

                # step5. Decoding to the original form
                df_after_enn_org = decode_to_original(df_after_enn, Categ, Num_disc, mapping_dict)

                # step6. Remove with LLM ("arcee-ai/trinity-large-preview:free")
                kept_index = run_filter_chunked(
                    df_after_enn_org,
                    orig_data,
                    data_desc,
                    column_desc,
                    original_sample_size=size,
                    filtering_chunk_size=size)
                if kept_index is not None and len(kept_index) > 0:
                    sub_gen_data = df_after_enn_org.loc[kept_index]
                    df_after_llm_org = pd.concat([df_after_llm_org, sub_gen_data], axis=0)

                    # step7. Re-Numericalization
                    df_after_llm = renumericalize_final_data(df_after_llm_org, Categ, Num_disc, Num_cont, mapping_dict)
                else:
                    print(f"Warning: Seed {current_loop_seed} produced no LLM-approved points.")    

            print(f"Random Seed {current_loop_seed} complete. Total Kept: {len(df_after_llm_org)}/{needed_num}")    
            current_loop_seed += 1

        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")

        # Final Result
        new_data = df_after_llm.iloc[:needed_num].reset_index(drop=True)
        time_str.append(elapsed) 
        yield_str.append(len(df_after_llm)/(len(df_extra)*current_loop_seed))
        kept_str.append(len(df_after_llm))
        over_df = pd.concat([df_val, new_data], axis=0)
        over_df.to_csv("Generated Data/{}/XEL_Trinity_Large/(delta={}){}_XEL_{}_full.csv".format(d_name,delta_value,d_name, Strategy[j]), index=False)     

## Random Oversampling (ROS) / SMOTE+ENN (SEN) / SMOTE+TOMEC (STM) - 70% 5CV

In [ ]:
# ROS

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "ROS_{}".format(Strategy[j]), "==========")
    over = RandomOverSampler(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            X_train, y_train = over.fit_resample(X_train, y_train) 
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_cv.append(elapsed)
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/ROS/{}_ROS_{}_{}th.csv".format(d_name,d_name,Strategy[j],n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/ROS/{}_ROS_time.csv".format(d_name,d_name), index=False)

In [ ]:
# SEN

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "SEN_{}".format(Strategy[j]), "==========")
    over = SMOTEENN(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            X_train, y_train = over.fit_resample(X_train, y_train) 
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_cv.append(elapsed)
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/SEN/{}_SEN_{}_{}th.csv".format(d_name,d_name,Strategy[j],n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/SEN/{}_SEN_time.csv".format(d_name,d_name), index=False)

In [ ]:
# STM

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "STM_{}".format(Strategy[j]), "==========")
    over = SMOTETomek(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        n_iter=0
        time_cv = []
        for train_index, val_index in skf.split(X, y):
            n_iter += 1
            print("=======", "{}th-cv".format(n_iter), "=======")
            X_train = X.iloc[train_index]
            y_train = y.iloc[train_index]
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
            majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
            minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)
            
            # Minority Generation
            start = time.time()
            X_train, y_train = over.fit_resample(X_train, y_train) 
            if n_iter == 1:
                print(list(y_train).count(0), list(y_train).count(1), len(y_train))
            elapsed = time.time() - start
            print(f"Run took {elapsed:.2f}s")
            time_cv.append(elapsed)
            over_df_raw = pd.concat([X_train, y_train], axis=1)
            over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
            over_df.to_csv("Generated Data/{}/STM/{}_STM_{}_{}th.csv".format(d_name,d_name,Strategy[j],n_iter), index=False)
        time_str.append(time_cv)
df_time = pd.DataFrame(np.array(time_str).T)
df_time.to_csv("Generated Data/{}/STM/{}_STM_time.csv".format(d_name,d_name), index=False)

## Random Oversampling (ROS) / SMOTE+ENN (SEN) / SMOTE+TOMEC (STM) - 30% test

In [ ]:
# ROS

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "ROS_{}".format(Strategy[j]), "==========")
    over = RandomOverSampler(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        X_over, y_over = over.fit_resample(X, y)  
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")
        time_str.append(elapsed)
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/ROS/{}_ROS_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)

In [ ]:
# SEN

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "SEN_{}".format(Strategy[j]), "==========")
    over = SMOTEENN(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        X_over, y_over = over.fit_resample(X, y)  
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")
        time_str.append(elapsed)
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/SEN/{}_SEN_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)

In [ ]:
# STM

In [ ]:
##################### Original Data Info #######################
print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

##################### Validation:Test = 70:30 #######################
df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
X = df_val.iloc[:, :-1]
y = df_val.iloc[:, -1]
print('<Train-Data Class Dist>\n', y.value_counts())
print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
##################### For Validation Set #######################
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
print("<min_strategy>:",min_strategy)

time_str = []
for j in range(len(Strategy)):
    print("==========", "STM_{}".format(Strategy[j]), "==========")
    over = SMOTETomek(sampling_strategy=Strategy[j], random_state=0)
    if min_strategy > Strategy[j]:
        time_str.append([0,0,0,0,0])
        continue         
    else:     
        majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
        minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
        needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
        print("Nedeed Samples:", needed_num)

        # Minority Generation
        start = time.time()
        X_over, y_over = over.fit_resample(X, y)  
        if n_iter == 1:
            print(list(y_over).count(0), list(y_over).count(1), len(y_over))
        elapsed = time.time() - start
        print(f"Run took {elapsed:.2f}s")
        time_str.append(elapsed)
        over_df_raw = pd.concat([X_train, y_train], axis=1)
        over_df = process_extra_samples(over_df_raw ,Num_cont, Num_disc, Categ, metadata)  # step3: snap&round
        over_df.to_csv("Generated Data/{}/STM/{}_STM_{}_full.csv".format(d_name,d_name,Strategy[j]), index=False)